# 03 — Generate Prior Belief Ensemble

Create a cloud of N parameter dictionaries sampled around the nominal textbook values. Each model in the ensemble represents one hypothesis about the true kinetic parameters.

**Output:** `workspace/prior_belief_cloud_N{N}_spread{S}.pkl`

In [1]:
import os
# Navigate to repo root so all relative paths resolve correctly.
if os.path.basename(os.getcwd()) in ('notebooks', 'examples'):
    os.chdir('..')

In [2]:
import pickle
import numpy as np
from active_learning.config import ActiveLearningConfig
from active_learning.utils import generate_dynamic_targets, generate_prior_ensemble

# Load nominal parts
with open(os.path.join("gcad", "data", "parts.pkl"), 'rb') as f:
    nominal_parts = pickle.load(f)

# Load selected circuits to auto-detect which parts are active
with open(os.path.join("data", "selected_M_circuits.pkl"), "rb") as f:
    circuits_list = pickle.load(f)

print(f"Loaded {len(nominal_parts)} nominal parts and {len(circuits_list)} circuits.")

Loaded 24 nominal parts and 3 circuits.


## Configure Prior

`spread_factor` controls the width of the prior. A value of 2.0 means parameters can range from 0.5x to 2.0x their nominal values.

In [3]:
config = ActiveLearningConfig(
    ensemble_size=200,
    spread_factor=2.0,
    dist_type='lognormal'
)

# Dynamically detect the active parts from the selected circuits
targets = generate_dynamic_targets(circuits_list)
print(f"Active targets: {targets}")

Active targets: {'Z6': [0, 1, 2], 'I13': [0]}


## Generate and Save Ensemble

In [4]:
np.random.seed(42)
belief_ensemble = generate_prior_ensemble(nominal_parts, targets, config)
print(f"Generated {len(belief_ensemble)} models.")

os.makedirs("workspace", exist_ok=True)
ensemble_path = f"workspace/prior_belief_cloud_N{config.ensemble_size}_spread{config.spread_factor}.pkl"
with open(ensemble_path, 'wb') as f:
    pickle.dump(belief_ensemble, f)
print(f"Saved to '{ensemble_path}'.")

Generated 200 models.
Saved to 'workspace/prior_belief_cloud_N200_spread2.0.pkl'.


## Sanity Check — Is the Ground Truth Inside the Prior Cloud?

In [5]:
with open(os.path.join("data", "ground_truth", "true_parts.pkl"), 'rb') as f:
    true_parts = pickle.load(f)

print("Boundary check:")
for part, indices in targets.items():
    for i in indices:
        if part in true_parts:
            nominal = nominal_parts[part][i]
            truth = true_parts[part][i]
            lo = nominal / config.spread_factor
            hi = nominal * config.spread_factor
            inside = lo <= truth <= hi
            print(f"  {part}[{i}]: truth={truth:.4f}, cloud=[{lo:.4f}, {hi:.4f}] -> {'INSIDE' if inside else 'OUTSIDE'}")

Boundary check:
  Z6[0]: truth=0.0150, cloud=[0.0108, 0.0431] -> INSIDE
  Z6[1]: truth=72.5000, cloud=[30.5878, 122.3510] -> INSIDE
  Z6[2]: truth=0.0365, cloud=[0.0183, 0.0731] -> INSIDE
  I13[0]: truth=0.0090, cloud=[0.0058, 0.0231] -> INSIDE
